In [1]:
import mysql.connector
import pandas as pd

# Connect to MySQL via XAMPP
conn = mysql.connector.connect(
    host='localhost',
    user='root',
    password=''  # XAMPP default has no password
)

print("Connected successfully!", conn)

Connected successfully! <mysql.connector.connection_cext.CMySQLConnection object at 0x00000142D028A660>


In [2]:
cursor = conn.cursor()

# Create database
cursor.execute("CREATE DATABASE IF NOT EXISTS healthcare_db")
cursor.execute("USE healthcare_db")

print("Database created and selected!")

Database created and selected!


In [3]:
# Load the dataset
df = pd.read_csv('../data/healthcare_dataset.csv')

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nData types:\n", df.dtypes)
df.head()

Shape: (55500, 15)

Columns: ['Name', 'Age', 'Gender', 'Blood Type', 'Medical Condition', 'Date of Admission', 'Doctor', 'Hospital', 'Insurance Provider', 'Billing Amount', 'Room Number', 'Admission Type', 'Discharge Date', 'Medication', 'Test Results']

Data types:
 Name                   object
Age                     int64
Gender                 object
Blood Type             object
Medical Condition      object
Date of Admission      object
Doctor                 object
Hospital               object
Insurance Provider     object
Billing Amount        float64
Room Number             int64
Admission Type         object
Discharge Date         object
Medication             object
Test Results           object
dtype: object


,Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results
0,Bobby JacksOn,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons and Miller,Blue Cross,18856.281306,328,Urgent,2024-02-02,Paracetamol,Normal
1,LesLie TErRy,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.327287,265,Emergency,2019-08-26,Ibuprofen,Inconclusive
2,DaNnY sMitH,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook PLC,Aetna,27955.096079,205,Emergency,2022-10-07,Aspirin,Normal
3,andrEw waTtS,28,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers and Vang,",Medicare,37909.782410,450,Elective,2020-12-18,Ibuprofen,Abnormal
4,adrIENNE bEll,43,Female,AB+,Cancer,2022-09-19,Kathleen Hanna,White-White,Aetna,14238.317814,458,Urgent,2022-10-09,Penicillin,Abnormal


In [4]:
cursor.execute("USE healthcare_db")

create_table = """
CREATE TABLE IF NOT EXISTS patients (
    id INT AUTO_INCREMENT PRIMARY KEY,
    name VARCHAR(100),
    age INT,
    gender VARCHAR(10),
    blood_type VARCHAR(5),
    medical_condition VARCHAR(100),
    date_of_admission DATE,
    doctor VARCHAR(100),
    hospital VARCHAR(100),
    insurance_provider VARCHAR(100),
    billing_amount FLOAT,
    room_number INT,
    admission_type VARCHAR(50),
    discharge_date DATE,
    medication VARCHAR(100),
    test_results VARCHAR(50)
)
"""

cursor.execute(create_table)
print("Table created successfully!")

Table created successfully!


In [5]:
# Clean column names for MySQL compatibility
df.columns = [col.lower().replace(' ', '_') for col in df.columns]

# Convert date columns
df['date_of_admission'] = pd.to_datetime(df['date_of_admission'])
df['discharge_date'] = pd.to_datetime(df['discharge_date'])

print("Columns cleaned:", df.columns.tolist())
print("Ready to load!")

Columns cleaned: ['name', 'age', 'gender', 'blood_type', 'medical_condition', 'date_of_admission', 'doctor', 'hospital', 'insurance_provider', 'billing_amount', 'room_number', 'admission_type', 'discharge_date', 'medication', 'test_results']
Ready to load!


In [6]:
# Insert rows into MySQL
insert_query = """
INSERT INTO patients (
    name, age, gender, blood_type, medical_condition,
    date_of_admission, doctor, hospital, insurance_provider,
    billing_amount, room_number, admission_type,
    discharge_date, medication, test_results
) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

# Convert dataframe to list of tuples
rows = list(df.itertuples(index=False, name=None))

# Execute bulk insert
cursor.executemany(insert_query, rows)
conn.commit()

print(f"Successfully loaded {cursor.rowcount} rows into MySQL!")

OperationalError: 2013 (HY000): Lost connection to MySQL server during query

In [7]:
# Reconnect with higher timeout
conn = mysql.connector.connect(
    host='localhost',
    user='root',
    password='',
    database='healthcare_db',
    connection_timeout=300
)
cursor = conn.cursor()

# Increase MySQL timeout
cursor.execute("SET GLOBAL net_write_timeout = 300")
cursor.execute("SET GLOBAL net_read_timeout = 300")
cursor.execute("SET GLOBAL wait_timeout = 300")

print("Reconnected with higher timeout!")

Reconnected with higher timeout!


In [8]:
# Convert to list of tuples
rows = list(df.itertuples(index=False, name=None))

# Insert in batches
batch_size = 1000
total = len(rows)

for i in range(0, total, batch_size):
    batch = rows[i:i + batch_size]
    cursor.executemany(insert_query, batch)
    conn.commit()
    print(f"Inserted rows {i} to {min(i+batch_size, total)} of {total}")

print("All rows inserted successfully!")

Inserted rows 0 to 1000 of 55500
Inserted rows 1000 to 2000 of 55500
Inserted rows 2000 to 3000 of 55500
Inserted rows 3000 to 4000 of 55500
Inserted rows 4000 to 5000 of 55500
Inserted rows 5000 to 6000 of 55500
Inserted rows 6000 to 7000 of 55500
Inserted rows 7000 to 8000 of 55500
Inserted rows 8000 to 9000 of 55500
Inserted rows 9000 to 10000 of 55500
Inserted rows 10000 to 11000 of 55500
Inserted rows 11000 to 12000 of 55500
Inserted rows 12000 to 13000 of 55500
Inserted rows 13000 to 14000 of 55500
Inserted rows 14000 to 15000 of 55500
Inserted rows 15000 to 16000 of 55500
Inserted rows 16000 to 17000 of 55500
Inserted rows 17000 to 18000 of 55500
Inserted rows 18000 to 19000 of 55500
Inserted rows 19000 to 20000 of 55500
Inserted rows 20000 to 21000 of 55500
Inserted rows 21000 to 22000 of 55500
Inserted rows 22000 to 23000 of 55500
Inserted rows 23000 to 24000 of 55500
Inserted rows 24000 to 25000 of 55500
Inserted rows 25000 to 26000 of 55500
Inserted rows 26000 to 27000 of 55

In [9]:
# Verify row count in MySQL
cursor.execute("SELECT COUNT(*) FROM patients")
result = cursor.fetchone()
print(f"Total rows in MySQL: {result[0]}")

# Preview data directly from MySQL
cursor.execute("SELECT * FROM patients LIMIT 5")
columns = [desc[0] for desc in cursor.description]
sample = cursor.fetchall()
df_preview = pd.DataFrame(sample, columns=columns)
df_preview

Total rows in MySQL: 55500


,id,name,age,gender,blood_type,medical_condition,date_of_admission,doctor,hospital,insurance_provider,billing_amount,room_number,admission_type,discharge_date,medication,test_results
0,1,Bobby JacksOn,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons and Miller,Blue Cross,18856.3,328,Urgent,2024-02-02,Paracetamol,Normal
1,2,LesLie TErRy,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.3,265,Emergency,2019-08-26,Ibuprofen,Inconclusive
2,3,DaNnY sMitH,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook PLC,Aetna,27955.1,205,Emergency,2022-10-07,Aspirin,Normal
3,4,andrEw waTtS,28,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers and Vang,",Medicare,37909.8,450,Elective,2020-12-18,Ibuprofen,Abnormal
4,5,adrIENNE bEll,43,Female,AB+,Cancer,2022-09-19,Kathleen Hanna,White-White,Aetna,14238.3,458,Urgent,2022-10-09,Penicillin,Abnormal


## Database Setup Summary

- Connected to MySQL via XAMPP (localhost)
- Created database: `healthcare_db`
- Created table: `patients` with 16 columns
- Loaded 55,500 rows via batch insert (1,000 rows/batch)
- Used ETL pipeline: CSV → Pandas → MySQL
- Fixed connection timeout by inserting in batches
- Next step: SQL analysis queries